# Building GPT

https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=7

Handling Input

In [1]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
print("length of dataset in characters: ", len(text))

In [3]:
print(text[:1000])

In [4]:
# all the unique chars that appear in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

In [5]:
# mapping from chars to ints

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s:[stoi[c] for c in s] # takes in a string, returns list of integers
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("test"))
print(decode(encode("bruh")))

In [6]:
# encode the entire text dataset as a torch.Tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

In [7]:
# Train and Validation Split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8
train_data[:block_size+1]

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

In [10]:
#torch.manual_seed(1337)
batch_size = 4 # number of independent sequences we process in parallel
block_size = 8 # maximum context length for predictions

data = train_data
print("Data Shape", data.shape)

ix = torch.randint(len(data) - block_size, (batch_size,))
inbt = [data[i:i+block_size] for i in ix]
x = torch.stack([data[i:i+block_size] for i in ix])
print(inbt)
print(x.shape)

In [11]:

torch.manual_seed(1337)
batch_size = 4 # number of independent sequences we process in parallel
block_size = 8 # maximum context length for predictions

def get_batch(split):
    # generate a small batch of data with inputs x and targets y
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # random offsets of the data
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

xb, yb = get_batch('train')
print('inputs')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

In [12]:
print(xb)

Creating a Bigram Language Model


In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [14]:
torch.manual_seed(1337);

In [15]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None): # idx and targets are both (B, T) tensor of integers
        # idx (4, 8)
        logits = self.token_embedding_table(idx)  #(4, 8, vocab_size) OR (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)

            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)

            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)

            # sample from distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)

        return idx

m = BigramLanguageModel(vocab_size)
logits,loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1, 1), dtype = torch.long) # batch of 1, time of 1, holding a zero (newline).
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist())) # [0] to unpluck the batch from the batch of 1

Adam Optimizer
Normally we do 3e-4, but for very small neural nets, we can get away with higher learning rates.

In [16]:
 # create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [17]:
batch_size = 32
for steps in range(15000):
    # sample a batch of data
    xb, yb = get_batch('train')

    #eval the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

In [18]:
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))

Mathematical Trick in Self Attention.

A token should only talk to previous context, not future context.

A simple way to couple them is to average its feature with all the previous tokens with the current token.

In [19]:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

In [20]:
# We want x[b, t] = mean_{i <= t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

print(xbow)

In [21]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

In [22]:
# version 2

wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x  # (T, T) @ (B, T, C) --> (B, T, T) @ (B, T, C) --> (B, T, C)
torch.allclose(xbow, xbow2, atol=1e-7)

In [23]:
# version 3: use softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T)) # like an interaction strength/affinity matrix, this will not be all zeros
wei = wei.masked_fill(tril==0, float('-inf')) #for all the elements where tril is 0, set wei to -inf, used so tokens from the past cannot communicate
print(wei)
wei = F.softmax(wei, dim=-1)
print(wei)
xbow3 = wei @ x
torch.allclose(xbow, xbow3, atol=1e-7)
print(xbow3)

Attention:
1:02:09

Every single token at each position will emit two vectors: a query and a key vector.

A query vector is (what am i looking for), key vector (what do i contain).

We calculate affinities by doing a dot product.



In [24]:
# version 4: self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) #(B, T, 16)
q = query(x) # (B, T, 16)
wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) --> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)


v = value(x) # (B, T, 16)
out = wei @ v  # (B, T, 16)
# out = wei @ x

out.shape

Attention is a communication mechanism. Each token looks at others through the query and value dot product and add value matrices to update its embedding with new information.

There is no notion of space as attention acts over a set of vectors, so we need to positionally encode tokens.

Each example across a batch dimension is processed independently (it's a batch of independent of examples).

In an "encoder" attention block, we don't do masking with `tril` to allow all tokens to communicate. We are doing a "decoder" attention block because it has triangular masking (used in autoregressive settings like language modeling.)

"self-attention" means keys, queries, and values are produced from the same source `x`. In principle, attention is more general than this, so the queries can be produced from `x` but the keys and the values could be produced from another external source (i.e. encoder module).

"Scaled" attention divides `wei` by $\frac{1}{\text{head size}}$. This makes it so when `Q`, `K` are unit variance, then `wei` will also be unit variance. It's important that at initialization, `wei` should be fairly diffuse as otherwise softmax will converge towards one hot vectors (as it will sharpen towards maximum values) and saturate towards only one vector.



Self attention is communication, and once the data has been gathered, they need to think on that data individually.

Residual Connections

Transform the data but have a skip connection with addition from the previous features.

The computation happens from top to bottom, and you have this residual pathway, and you are free to fork off to do so computation and do addition.

This is useful because the gradients are distributed equally during addition.

The residual blocks are usually initialized in the beginning so they contribute very little to the pathway ( like not there) but during optimization, they come online over time.


In [38]:
class BatchNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        # parameters (trained with backprop)   (scale and shift)
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        # calculate the forward pass
        xmean = x.mean(1, keepdim = True) # batch mean
        xvar = x.var(1, keepdim = True) # batch variance

        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

torch.manual_seed(1337)
module = BatchNorm1d(100)
x = torch.randn(32, 100) # batch size 32
x = module(x)
x.shape

Layernorm normalizes the row instead of columns (change dim 0 from batchnorm to dim 1 for layernorm).

In [39]:
x[:,0].mean(), x[:,0].std() # one feature across all batch inputs

In [40]:
x[0,:].mean(), x[0,:].std() # mean,std of single input from batch of all features

Since computation does not span across examples, we don't need buffers to hold the running_mean/var of the inputs.

More common to apply layer norm before transformation.

Dropout is something you can add right before the connection before the residual pathway.

Every forward backward pass, dropout randomly shuts off some subset of neurons and trains without them.

Because the mask of neurons is changed every forward backward pass, it trains an ensemble of subnetworks. At test time, everything is fully enabled and the subnetworks are merged into a single ensemble. Regularlization technique.

Usually all the keys, query, value matrices are concatenated together originally instead of decoupled.

GELU:
train loss 0.9774, val loss 1.5096